# Report figures and numbers (Lab 4, run 2)

A copy of `Lab_4_AVCV_2026_2027_Students.ipynb` with the demo cells removed. It loads the
**submitted run-2 checkpoints** (`custom_best.pt`, `ft_best.pt`) and writes everything the
report reads into `FIG_DIR`:

| File | Content |
|---|---|
| `val_numbers.tex` | validation macro F1 (plain / TTA / TTA + calibration) and accuracy macros |
| `per_class_rows.tex` | per-class precision / recall / F1 rows for both models |
| `fig_confusion.pdf` | row-normalised validation confusion matrices |
| `fig_errors.pdf` | most confident validation errors for the most frequent confusions |
| `fig_curves.pdf` | loss and macro-F1 curves (**only** with `RETRAIN_FOR_CURVES = True`) |
| `metrics.json` | all of the above as numbers |

Copy the files into `report/figures/` and recompile: `main.tex` picks them up automatically.

**Notes for the report.** The calibrated validation F1 is fitted *and* scored on the
validation split, so it is optimistic; only the CodaBench scores are unbiased. Curves from a
retrain describe a re-run of the same recipe, not the exact submitted weights.

It does not write submission files and never overwrites the submitted checkpoints.

In [ ]:
%matplotlib inline

In [ ]:
from pathlib import Path
import copy
import numbers
import random
import time
import zipfile

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from PIL import Image
from skimage import io, transform, util

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, utils, models

# Install missing dependencies in your chosen environment before running.
# In a notebook, you can use:
# %pip install torch torchvision pandas scikit-image matplotlib scikit-learn pillow

SEED = 42
CLASS_NAMES = ['Water', 'Very Low', 'Low', 'Moderate',
               'High', 'Very High', 'Non-burnable']
NUM_CLASSES = len(CLASS_NAMES)
MAX_TRAIN_SAMPLES = 0  # Set to 0 to use the full training split.
CUSTOM_EPOCHS = 50        # Run 2 (test macro F1 0.49) used 50
FT_EPOCHS = 20            # Run 2 (test macro F1 0.50) used 20
CUSTOM_MAX_LR = 3e-3      # OneCycle peak learning rate for the from-scratch net
FT_BACKBONE_LR = 1e-4     # Pretrained ConvNeXt layers
FT_HEAD_LR = 1e-3         # Newly initialised 7-class head
TRAIN_CROP = 288          # Random training crop; validation/test use the full 320 px image
BATCH_SIZE = 64           # Lower to 32 if the GPU runs out of memory
USE_TTA = True            # Average predictions over the 8 flip/rot90 views
CALIBRATE = True          # Per-class probability weights tuned on validation for macro F1
LOAD_CHECKPOINTS = False  # True: skip training and load the saved best weights (reproduction)

# Smoke test settings:
# MAX_TRAIN_SAMPLES = 128   # full run: 0
# CUSTOM_EPOCHS = 1         # full run: 50
# FT_EPOCHS = 1             # full run: 20

# Note: PIXEL_MEAN and PIXEL_STD come directly from AlexNet's
# documentation in PyTorch:
# https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.alexnet.html
PIXEL_MEAN = [0.485, 0.456, 0.406]
PIXEL_STD = [0.229, 0.224, 0.225]

DATA_ARCHIVE = Path('db.zip')
DATA_ROOT = Path('data')
DB_DIR = DATA_ROOT / 'db'
OUTPUT_DIR = Path('outputs') / 'Lab_4_2026_2027'

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

seed_everything()
device = torch.device('cuda:0' if torch.cuda.is_available() else
                      'mps' if torch.backends.mps.is_available() else 'cpu')
# Mixed precision only on CUDA; worker processes only where fork() is available (Linux/Colab).
USE_AMP = device.type == 'cuda'
NUM_WORKERS = 2 if device.type == 'cuda' else 0
print(f'Device: {device} | AMP: {USE_AMP} | workers: {NUM_WORKERS}')
print(f'Class order: {CLASS_NAMES}')

# ---------------------------------------------------------------------------
# Report-figure settings (this notebook only)
# ---------------------------------------------------------------------------
LOAD_CHECKPOINTS = True     # Figures and numbers come from the SUBMITTED run-2 weights.
RETRAIN_FOR_CURVES = False  # True: retrain both models only to record loss/F1 curves.
                            # The run-2 checkpoints hold weights only, so curves cannot be
                            # recovered from them. A retrain writes *_rerun.pt and never
                            # overwrites custom_best.pt / ft_best.pt. GPU strongly recommended.
SMOKE = False               # True: tiny local pipeline check; outputs are NOT report material.
if SMOKE:
    # >10 optimiser steps: OneCycle's warm-up phase breaks (NaN lr) with fewer.
    MAX_TRAIN_SAMPLES, CUSTOM_EPOCHS, FT_EPOCHS = 384, 2, 2
    RETRAIN_FOR_CURVES = True
FIG_DIR = OUTPUT_DIR / 'report_figures'


In [ ]:
# Google Colab setup: mount Google Drive and read db.zip from the project folder.
# Drive folder: https://drive.google.com/drive/folders/1A5iKDwVFsfKCle7GFq-_h1SrNtYqpYDP
# Outside Colab, the local db.zip in the working directory is used instead.
import shutil

DRIVE_DIR = None
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    DRIVE_DIR = Path('/content/drive/MyDrive/lab_4_CV')
    DATA_ARCHIVE = DRIVE_DIR / 'db.zip'
    if not DATA_ARCHIVE.is_file():
        raise FileNotFoundError(f'Archive not found: {DATA_ARCHIVE}')
    print(f'Using Google Drive archive: {DATA_ARCHIVE}')
elif DATA_ARCHIVE.is_file():
    print(f'Using local archive: {DATA_ARCHIVE.resolve()}')
elif (DB_DIR / 'train_reduced.csv').is_file():
    print(f'Using already extracted dataset: {DB_DIR.resolve()}')
else:
    raise FileNotFoundError('Place db.zip in the working directory, or run in Google Colab.')

In [ ]:
required_csvs = [DB_DIR / f'{split}_reduced.csv'
                 for split in ('train', 'val', 'test')]

if DATA_ARCHIVE.is_file():
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    restored_files = 0
    with zipfile.ZipFile(DATA_ARCHIVE) as archive:

        for member in archive.infolist():
            if not member.filename.startswith('db/'):
                continue
            target = DATA_ROOT / member.filename
            if not member.is_dir() and target.is_file() and target.stat().st_size == member.file_size:
                continue
            if not target.resolve().is_relative_to(DATA_ROOT.resolve()):
                raise ValueError('The archive contains an invalid dataset path.')
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            elif not target.is_file() or target.stat().st_size != member.file_size:
                archive.extract(member, DATA_ROOT)
                restored_files += 1
    print(f'Dataset ready at {DB_DIR.resolve()}; restored {restored_files} files.')

if not all(path.is_file() for path in required_csvs):
    raise FileNotFoundError(
        'Dataset CSV files were not found. Start the kernel in the notebook directory '
        'or update DATA_ARCHIVE and DATA_ROOT in the setup cell.')

for csv_path in required_csvs:
    ids = pd.read_csv(csv_path, usecols=['id'], dtype={'id': str})['id']
    for image_id in ids:
        image_path = DB_DIR / 'images' / f'{image_id}.jpg'
        if not image_path.is_file() or image_path.stat().st_size == 0:
            raise FileNotFoundError(
                f'Missing or empty image: {image_path}. Supply the original db.zip '
                'and rerun this setup cell to restore it.')

print('All indexed images exist and are nonempty.')

In [ ]:
from concurrent.futures import ThreadPoolExecutor


def _read_rgb_uint8(path):
    """Decode one image as an H x W x 3 uint8 array."""
    image = io.imread(path)
    if image.ndim == 2:
        image = np.repeat(image[..., None], 3, axis=2)
    elif image.ndim == 3 and image.shape[2] == 4:
        image = image[..., :3]
    if image.ndim != 3 or image.shape[2] != 3:
        raise ValueError(f'Expected an RGB image with three channels: {path}')
    return util.img_as_ubyte(image)


class FireRiskDataset(Dataset):
    """Read RGB images and integer labels using a CSV index.

    Args:
        csv_file: CSV with columns 'id' and 'label'.
        root_dir: Dataset directory containing the 'images' subdirectory.
        transform: Optional callable that receives and returns a sample dictionary.
        maxSize: Maximum subset size; 0 selects the full split.
        cache: Decode every image once and keep it in RAM as uint8 (about 0.3 MB
            per image). This removes JPEG decoding from every training epoch.
    """

    def __init__(self, csv_file, root_dir, transform=None, maxSize=0, cache=False):

        if not isinstance(maxSize, numbers.Integral) or maxSize < 0:
            raise ValueError('maxSize must be a nonnegative integer.')

        self.dataset = pd.read_csv(csv_file, dtype={'id': str, 'label': int})

        if not {'id', 'label'}.issubset(self.dataset.columns) or self.dataset.empty:
            raise ValueError('The CSV must contain nonempty id and label columns.')
        if not self.dataset['label'].isin(range(-1, NUM_CLASSES)).all():
            raise ValueError('Labels must be -1 (hidden test labels) or 0 through 6.')
        if maxSize > 0:
            indices = np.random.RandomState(SEED).permutation(len(self.dataset))
            self.dataset = self.dataset.iloc[indices[:maxSize]].reset_index(drop=True)

        self.img_dir = Path(root_dir) / 'images'
        self.transform = transform
        self.classes = CLASS_NAMES.copy()
        self.labels = self.dataset['label'].to_numpy()
        self.images = None
        if cache:
            paths = [self.img_dir / f'{image_id}.jpg' for image_id in self.dataset['id']]
            with ThreadPoolExecutor(max_workers=8) as pool:
                self.images = list(pool.map(_read_rgb_uint8, paths))

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):

        if torch.is_tensor(idx):
            idx = idx.item()
        idx = int(idx)
        if self.images is not None:
            image = self.images[idx]
        else:
            image = _read_rgb_uint8(self.img_dir / f"{self.dataset.iloc[idx]['id']}.jpg")
        sample = {'image': util.img_as_float32(image), 'label': int(self.labels[idx])}

        return self.transform(sample) if self.transform is not None else sample

In [ ]:
def _crop_size(output_size):

    if isinstance(output_size, numbers.Integral):
        output_size = (int(output_size), int(output_size))

    if (not isinstance(output_size, tuple) or len(output_size) != 2 or
        any(not isinstance(v, numbers.Integral) or v <= 0 for v in output_size)):
        raise ValueError('Size must be a positive integer or a pair of positive integers.')

    return tuple(int(v) for v in output_size)

class Rescale:
    """Resize to (height, width), or resize the shorter side preserving aspect ratio."""

    def __init__(self, output_size):
        self.output_size = (int(output_size) if isinstance(output_size, numbers.Integral)
                            else _crop_size(output_size))
        if isinstance(self.output_size, int) and self.output_size <= 0:
            raise ValueError('The output size must be positive.')

    def __call__(self, sample):
        image, label = sample['image'], sample['label']
        h, w = image.shape[:2]
        if isinstance(self.output_size, int):
            scale = self.output_size / min(h, w)
            new_h, new_w = round(h * scale), round(w * scale)
        else:
            new_h, new_w = self.output_size
        image = transform.resize(image, (new_h, new_w), anti_aliasing=True)

        return {'image': image.astype(np.float32), 'label': label}

class RandomCrop:
    """Choose uniformly among all valid crops, including the final row/column."""

    def __init__(self, output_size):
        self.output_size = _crop_size(output_size)

    def __call__(self, sample):

        image, label = sample['image'], sample['label']
        h, w = image.shape[:2]
        new_h, new_w = self.output_size

        if new_h > h or new_w > w:
            raise ValueError('Crop exceeds image dimensions; resize the image first.')
        top = np.random.randint(0, h - new_h + 1)
        left = np.random.randint(0, w - new_w + 1)

        return {'image': image[top:top + new_h, left:left + new_w], 'label': label}

class CenterCrop:
    """Take a deterministic central crop without silently changing its size."""

    def __init__(self, output_size):
        self.output_size = _crop_size(output_size)

    def __call__(self, sample):

        image, label = sample['image'], sample['label']
        h, w = image.shape[:2]
        new_h, new_w = self.output_size

        if new_h > h or new_w > w:
            raise ValueError('Crop exceeds image dimensions; resize the image first.')
        top, left = (h - new_h) // 2, (w - new_w) // 2

        return {'image': image[top:top + new_h, left:left + new_w], 'label': label}

class ToTensor:
    """Convert HWC images to contiguous float32 CHW tensors and labels to int64."""

    def __call__(self, sample):
        image = np.ascontiguousarray(sample['image'].transpose(2, 0, 1), dtype=np.float32)
        return {'image': torch.from_numpy(image),
                'label': torch.tensor(sample['label'], dtype=torch.long)}

class Normalize:
    """Apply per-channel normalisation without mutating the input tensor."""

    def __init__(self, mean, std):

        self.mean = np.asarray(mean, dtype=np.float32)
        self.std = np.asarray(std, dtype=np.float32)

        if (self.mean.ndim != 1 or self.std.shape != self.mean.shape or
            not np.isfinite(self.mean).all() or not np.isfinite(self.std).all() or
            not (self.std > 0).all()):
            raise ValueError('Mean/std must be matching finite vectors with positive std.')

    def __call__(self, sample):

        image = sample['image']

        if image.ndim != 3 or image.shape[0] != len(self.mean):
            raise ValueError('The CHW tensor and normalisation vectors must match.')

        mean = torch.as_tensor(self.mean, dtype=image.dtype, device=image.device)[:, None, None]
        std = torch.as_tensor(self.std, dtype=image.dtype, device=image.device)[:, None, None]
        return {'image': (image - mean) / std, 'label': sample['label']}

In [ ]:
class ApplyToImage:
    """Apply an image-only transform while preserving the sample's other fields.

    For the tensor transforms below, use this wrapper after ToTensor() and
    before Normalize(). ColorJitter expects the image to still be in [0, 1].
    """

    def __init__(self, transform):
        self.transform = transform

    def __call__(self, sample):
        return {**sample, 'image': self.transform(sample['image'])}


class RandomRotate90:
    """Rotate a CHW tensor by a uniformly sampled 0, 90, 180 or 270 degrees.

    No interpolation or corner padding is needed. Use square inputs to keep
    the same height and width for batching.
    """

    def __call__(self, image):
        k = torch.randint(0, 4, ()).item()
        return torch.rot90(image, k, dims=(-2, -1)).contiguous()


In [ ]:
def image_for_display(tensor):
    """Undo normalisation for display; these RGB pixels are not model inputs."""
    image = tensor.detach().cpu()
    mean = torch.tensor(PIXEL_MEAN, dtype=image.dtype)[:, None, None]
    std = torch.tensor(PIXEL_STD, dtype=image.dtype)[:, None, None]
    return (image * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

In [ ]:
train_transform = transforms.Compose([
    RandomCrop(TRAIN_CROP),
    ToTensor(),
    ApplyToImage(transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        RandomRotate90(),
        transforms.ColorJitter(
            brightness=0.1, contrast=0.1, saturation=0.05,
        ),
    ])),
    Normalize(PIXEL_MEAN, PIXEL_STD),
])


In [ ]:
eval_transform = transforms.Compose([ToTensor(), Normalize(PIXEL_MEAN, PIXEL_STD)])

since = time.time()
train_dataset = FireRiskDataset(
    DB_DIR / 'train_reduced.csv', DB_DIR, maxSize=MAX_TRAIN_SAMPLES,
    transform=train_transform, cache=True)
val_dataset = FireRiskDataset(DB_DIR / 'val_reduced.csv', DB_DIR,
                              transform=eval_transform, cache=True)
test_dataset = FireRiskDataset(DB_DIR / 'test_reduced.csv', DB_DIR,
                               transform=eval_transform, cache=True)
print(f'Decoded and cached {len(train_dataset) + len(val_dataset) + len(test_dataset)} '
      f'images in {time.time() - since:.0f}s')

for i in range(min(4, len(train_dataset))):
    sample = train_dataset[i]
    print(i, sample['image'].shape, int(sample['label']))
print('Validation image shape:', tuple(val_dataset[0]['image'].shape))
print(f'Training subset size: {len(train_dataset)}')

In [ ]:
def seed_worker(worker_id):
    # PyTorch assigns a worker seed; also seed NumPy and Python random.
    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

train_generator = torch.Generator().manual_seed(SEED)
loader_options = dict(num_workers=NUM_WORKERS, worker_init_fn=seed_worker,
                      pin_memory=device.type == 'cuda',
                      persistent_workers=NUM_WORKERS > 0)
train_dataloader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, generator=train_generator,
    **loader_options)
val_dataloader = DataLoader(val_dataset, batch_size=64, shuffle=False, **loader_options)
test_dataloader = DataLoader(test_dataset, batch_size=64, shuffle=False, **loader_options)


In [ ]:
class ResidualBlock(nn.Module):
    """Basic residual block: two 3x3 conv/BN layers plus a shortcut connection.

    When the stride or channel count changes, the shortcut is a 1x1 conv + BN so
    the two branches can be added. The last BN starts at zero, so each block
    begins as an identity mapping, which makes the deeper net easier to train.
    """

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        nn.init.zeros_(self.bn2.weight)
        self.shortcut = nn.Identity()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x), inplace=True)


class CustomNet(nn.Module):
    """Small residual CNN trained from scratch: N x 3 x H x W -> N x 7 raw logits."""

    def __init__(self, widths=(32, 64, 128, 256), blocks_per_stage=2, dropout=0.3):
        super().__init__()
        # Stem halves the resolution: 288 -> 144 (training) or 320 -> 160 (evaluation).
        self.stem = nn.Sequential(
            nn.Conv2d(3, widths[0], 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(widths[0]),
            nn.ReLU(inplace=True),
        )
        stages, in_channels = [], widths[0]
        for i, width in enumerate(widths):
            for j in range(blocks_per_stage):
                stride = 2 if (i > 0 and j == 0) else 1
                stages.append(ResidualBlock(in_channels, width, stride))
                in_channels = width
        # Spatial sizes at 288 px: 144 -> 144 -> 72 -> 36 -> 18 across the four stages.
        self.features = nn.Sequential(*stages)
        # Global average pooling makes the classifier independent of the input size.
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(in_channels, NUM_CLASSES),
        )
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')

    def forward(self, x):
        return self.classifier(self.features(self.stem(x)))  # N x 7 logits

In [ ]:
seed_everything()

customNet = CustomNet().to(device)

data = next(iter(train_dataloader))
inputs = data['image'].to(device, dtype=torch.float32)

print(f'Input batch shape: {tuple(inputs.shape)}')

with torch.no_grad():
    outputs = customNet(inputs)
if outputs.shape != (inputs.shape[0], NUM_CLASSES):
    raise ValueError('CustomNet must produce one seven-score vector per image.')

print(f'Output batch shape: {tuple(outputs.shape)}')
print(f'CustomNet parameters: {sum(p.numel() for p in customNet.parameters()) / 1e6:.2f}M')

In [ ]:
from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
train_label_counts = np.bincount(train_dataset.labels, minlength=NUM_CLASSES)
class_weights = 1 / np.sqrt(np.maximum(train_label_counts, 1))
class_weights = torch.tensor(class_weights / class_weights.mean(), dtype=torch.float32)
print('Class weights:', np.round(class_weights.numpy(), 3))

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
optimizer_ft = optim.AdamW(customNet.parameters(), lr=CUSTOM_MAX_LR, weight_decay=0.05)
exp_lr_scheduler = lr_scheduler.OneCycleLR(
    optimizer_ft, max_lr=CUSTOM_MAX_LR, epochs=CUSTOM_EPOCHS,
    steps_per_epoch=len(train_dataloader), pct_start=0.2)

In [ ]:
def train_model(model, criterion, optimizer, scheduler, num_epochs=25,
                plot_confusion_matrix=True, step_per_batch=False, checkpoint_path=None):
    """Run fixed-budget training and restore the best validation-macro-F1 model.

    Returns (model, history), where history holds per-epoch losses, macro F1 and lr.
    """

    if not isinstance(num_epochs, numbers.Integral) or num_epochs < 1:
        raise ValueError('num_epochs must be a positive integer.')
    if not isinstance(criterion, nn.CrossEntropyLoss) or criterion.reduction != 'mean':
        raise ValueError('Use CrossEntropyLoss with reduction="mean".')
    if any(len(dataloaders[phase].dataset) == 0 for phase in ('train', 'val')):
        raise ValueError('Training and validation datasets must be nonempty.')

    seed_everything()

    if dataloaders['train'].generator is not None:
        dataloaders['train'].generator.manual_seed(SEED)

    criterion = criterion.to(device)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)

    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_f1 = -float('inf')
    best_epoch = -1
    best_predictions = best_labels = None
    labels_order = np.arange(NUM_CLASSES)
    history = {'train_loss': [], 'train_f1': [], 'val_loss': [], 'val_f1': [], 'lr': []}

    for epoch in range(num_epochs):
        print(f'Epoch {epoch}/{num_epochs - 1}  (lr {optimizer.param_groups[-1]["lr"]:.2e})')
        print('-' * 10)
        history['lr'].append(optimizer.param_groups[-1]['lr'])

        for phase in ('train', 'val'):
            model.train(phase == 'train')
            running_loss = 0.0
            loss_denominator = 0.0
            phase_predictions, phase_labels = [], []

            for sample in dataloaders[phase]:
                inputs = sample['image'].to(device, dtype=torch.float32, non_blocking=True)
                labels = sample['label'].to(device, dtype=torch.long, non_blocking=True)

                if phase == 'train':
                    optimizer.zero_grad(set_to_none=True)
                with torch.set_grad_enabled(phase == 'train'), \
                        torch.autocast(device.type, dtype=torch.float16, enabled=USE_AMP):
                    logits = model(inputs)
                    if logits.shape != (labels.numel(), NUM_CLASSES):
                        raise ValueError('The network must return N x 7 logits.')
                    loss = criterion(logits.float(), labels)
                if not torch.isfinite(loss):
                    raise ValueError('Non-finite loss; check inputs and optimiser settings.')
                if phase == 'train':
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()
                    if step_per_batch and scheduler is not None:
                        scheduler.step()

                denominator = (labels.numel() if criterion.weight is None else
                               criterion.weight[labels].sum().item())
                running_loss += loss.item() * denominator
                loss_denominator += denominator
                phase_predictions.append(logits.detach().argmax(dim=1).cpu().numpy())
                phase_labels.append(labels.detach().cpu().numpy())

            if phase == 'train' and scheduler is not None and not step_per_batch:
                scheduler.step()
            predictions = np.concatenate(phase_predictions)
            true_labels = np.concatenate(phase_labels)
            epoch_loss = running_loss / loss_denominator
            epoch_f1 = f1_score(true_labels, predictions, labels=labels_order,
                                average='macro', zero_division=0)
            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_f1'].append(epoch_f1)
            print(f'{phase} Loss: {epoch_loss:.4f} Macro F1-score: {epoch_f1:.4f}')
            if phase == 'val' and epoch_f1 > best_f1:
                best_f1, best_epoch = epoch_f1, epoch
                best_model_wts = copy.deepcopy(model.state_dict())
                best_predictions = predictions.copy()
                best_labels = true_labels.copy()
                if checkpoint_path is not None:
                    Path(checkpoint_path).parent.mkdir(parents=True, exist_ok=True)
                    torch.save(best_model_wts, checkpoint_path)
        print()

    model.load_state_dict(best_model_wts)
    model.eval()

    elapsed = time.time() - since
    print(f'Training complete in {int(elapsed // 60)}m {int(elapsed % 60)}s')
    print(f'Best validation macro F1-score: {best_f1:.6f} (epoch {best_epoch})')
    if checkpoint_path is not None:
        print(f'Best weights saved to {checkpoint_path}')

    if plot_confusion_matrix:
        cm = confusion_matrix(best_labels, best_predictions, labels=labels_order,
                              normalize='true')
        display = ConfusionMatrixDisplay(100 * cm, display_labels=CLASS_NAMES)
        display.plot(xticks_rotation='vertical', values_format='.1f')
        plt.title('Row-normalised confusion matrix (%)')
        plt.tight_layout()
        plt.show()

    return model, history


def plot_history(history, title):
    """Plot loss and macro-F1 learning curves (for the report)."""
    epochs = np.arange(len(history['train_loss']))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for phase in ('train', 'val'):
        axes[0].plot(epochs, history[f'{phase}_loss'], label=phase)
        axes[1].plot(epochs, history[f'{phase}_f1'], label=phase)
    axes[0].set_title(f'{title}: loss')
    axes[1].set_title(f'{title}: macro F1')
    for ax in axes:
        ax.set_xlabel('Epoch')
        ax.grid(alpha=0.3)
        ax.legend()
    fig.tight_layout()
    plt.show()

In [ ]:
image_datasets = {'train': train_dataset, 'val': val_dataset}

dataloaders = {'train': train_dataloader, 'val': val_dataloader}

dataset_sizes = {phase: len(dataset) for phase, dataset in image_datasets.items()}
class_names = CLASS_NAMES.copy()

# Checkpoints go straight to Google Drive on Colab so they survive a disconnect.
CHECKPOINT_DIR = (DRIVE_DIR / 'outputs' / OUTPUT_DIR.name) if DRIVE_DIR is not None else OUTPUT_DIR
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CUSTOM_CKPT = CHECKPOINT_DIR / 'custom_best.pt'
FT_CKPT = CHECKPOINT_DIR / 'ft_best.pt'
print(f'Checkpoints: {CUSTOM_CKPT}, {FT_CKPT}')

CUSTOM_RERUN_CKPT = CHECKPOINT_DIR / 'custom_rerun.pt'
FT_RERUN_CKPT = CHECKPOINT_DIR / 'ft_rerun.pt'
if DRIVE_DIR is not None:
    FIG_DIR = CHECKPOINT_DIR / 'report_figures'   # survives a Colab disconnect
FIG_DIR.mkdir(parents=True, exist_ok=True)
print(f'Report figures: {FIG_DIR.resolve()}')


In [ ]:
import json

custom_history = None
if RETRAIN_FOR_CURVES:
    # Same recipe and seed as run 2, but a GPU retrain is not bit-identical to the submission.
    customNet, custom_history = train_model(
        customNet, criterion, optimizer_ft, exp_lr_scheduler, num_epochs=CUSTOM_EPOCHS,
        step_per_batch=True, checkpoint_path=CUSTOM_RERUN_CKPT, plot_confusion_matrix=False)
    (FIG_DIR / 'history_custom.json').write_text(json.dumps(custom_history))
elif (FIG_DIR / 'history_custom.json').is_file():
    custom_history = json.loads((FIG_DIR / 'history_custom.json').read_text())
    print('Loaded curves from an earlier retrain:', FIG_DIR / 'history_custom.json')

if CUSTOM_CKPT.is_file():
    customNet.load_state_dict(torch.load(CUSTOM_CKPT, map_location=device))
    custom_source = 'submitted run-2 checkpoint'
elif SMOKE:
    custom_source = 'SMOKE-TEST WEIGHTS (not the submission)'
else:
    raise FileNotFoundError(f'Submitted checkpoint not found: {CUSTOM_CKPT}')
customNet.eval()
print('CustomNet weights used for the figures:', custom_source)

In [ ]:
# This category uses ImageNet pretraining; do not disable it for a final submission.
seed_everything()

# ConvNeXt-Tiny (ImageNet-1K) replaces AlexNet: a much stronger modern backbone that
# expects the same 224 px input and ImageNet mean/std already used by the pipeline.
ft_weights = models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1
# Stochastic depth randomly skips residual blocks during training (regularisation);
# torchvision's default for Tiny is 0.1, raised slightly for this small dataset.
ftNet = models.convnext_tiny(weights=ft_weights, stochastic_depth_prob=0.2)

# The classifier is [LayerNorm2d, Flatten, Linear(768, 1000)]; replace the final layer
# with a newly initialised seven-class head.
num_features = ftNet.classifier[2].in_features
ftNet.classifier[2] = nn.Linear(num_features, NUM_CLASSES)
ftNet = ftNet.to(device)

num_params = sum(p.numel() for p in ftNet.parameters())
print(f'{type(ftNet).__name__}: {num_params / 1e6:.1f}M parameters')
print(ftNet.classifier)

In [ ]:
head_params = list(ftNet.classifier.parameters())
head_ids = {id(p) for p in head_params}
backbone_params = [p for p in ftNet.parameters() if id(p) not in head_ids]

optimizer_ft = optim.AdamW([
    {'params': backbone_params, 'lr': FT_BACKBONE_LR},
    {'params': head_params, 'lr': FT_HEAD_LR},
], weight_decay=0.05)
exp_lr_scheduler = lr_scheduler.OneCycleLR(
    optimizer_ft, max_lr=[FT_BACKBONE_LR, FT_HEAD_LR], epochs=FT_EPOCHS,
    steps_per_epoch=len(train_dataloader), pct_start=0.1)

In [ ]:
import json

ft_history = None
if RETRAIN_FOR_CURVES:
    # Same recipe and seed as run 2, but a GPU retrain is not bit-identical to the submission.
    ftNet, ft_history = train_model(
        ftNet, criterion, optimizer_ft, exp_lr_scheduler, num_epochs=FT_EPOCHS,
        step_per_batch=True, checkpoint_path=FT_RERUN_CKPT, plot_confusion_matrix=False)
    (FIG_DIR / 'history_ft.json').write_text(json.dumps(ft_history))
elif (FIG_DIR / 'history_ft.json').is_file():
    ft_history = json.loads((FIG_DIR / 'history_ft.json').read_text())
    print('Loaded curves from an earlier retrain:', FIG_DIR / 'history_ft.json')

if FT_CKPT.is_file():
    ftNet.load_state_dict(torch.load(FT_CKPT, map_location=device))
    ft_source = 'submitted run-2 checkpoint'
elif SMOKE:
    ft_source = 'SMOKE-TEST WEIGHTS (not the submission)'
else:
    raise FileNotFoundError(f'Submitted checkpoint not found: {FT_CKPT}')
ftNet.eval()
print('ConvNeXt-Tiny weights used for the figures:', ft_source)

In [ ]:
DIHEDRAL_VIEWS = [(k, flip) for k in range(4) for flip in (False, True)]


def predict_proba(model, loader, tta=USE_TTA):
    """Return (probabilities, labels) in loader order; optionally average 8 TTA views."""
    model.eval()
    probs, labels = [], []
    views = DIHEDRAL_VIEWS if tta else [(0, False)]
    with torch.inference_mode(), torch.autocast(device.type, dtype=torch.float16, enabled=USE_AMP):
        for sample in loader:
            inputs = sample['image'].to(device, dtype=torch.float32, non_blocking=True)
            batch_probs = 0
            for k, flip in views:
                view = torch.flip(inputs, dims=(-1,)) if flip else inputs
                view = torch.rot90(view, k, dims=(-2, -1))
                logits = model(view).float()
                if logits.shape != (inputs.shape[0], NUM_CLASSES):
                    raise ValueError('The network must return N x 7 logits.')
                batch_probs = batch_probs + torch.softmax(logits, dim=1)
            probs.append((batch_probs / len(views)).cpu().numpy())
            labels.append(sample['label'].numpy())
    if not probs:
        raise ValueError('The split must be nonempty.')
    return np.concatenate(probs).astype(np.float64), np.concatenate(labels)


def macro_f1(probs, labels):
    return f1_score(labels, probs.argmax(axis=1), labels=np.arange(NUM_CLASSES),
                    average='macro', zero_division=0)


def apply_class_weights(probs, weights):
    """Rescale class probabilities and renormalise each row to sum to 1."""
    scaled = probs * weights[None, :]
    return scaled / scaled.sum(axis=1, keepdims=True)


def fit_class_weights(probs, labels, grid=np.linspace(0.5, 3.0, 26), passes=3):
    """Coordinate ascent on per-class multipliers to maximise macro F1."""
    weights = np.ones(NUM_CLASSES)
    best = macro_f1(probs, labels)
    for _ in range(passes):
        for c in range(NUM_CLASSES):
            for value in grid:
                trial = weights.copy()
                trial[c] = value
                score = macro_f1(apply_class_weights(probs, trial), labels)
                if score > best + 1e-9:
                    best, weights = score, trial
    return weights, best


def assess_and_predict(model, name):
    """Report validation macro F1 (plain / TTA / calibrated) and return test probabilities."""
    val_plain, val_labels = predict_proba(model, val_dataloader, tta=False)
    val_probs, _ = predict_proba(model, val_dataloader, tta=USE_TTA)
    test_probs, _ = predict_proba(model, test_dataloader, tta=USE_TTA)
    weights = np.ones(NUM_CLASSES)
    if CALIBRATE:
        weights, _ = fit_class_weights(val_probs, val_labels)
    val_final = apply_class_weights(val_probs, weights)
    print(f'{name}: val macro F1  plain {macro_f1(val_plain, val_labels):.4f} | '
          f'TTA {macro_f1(val_probs, val_labels):.4f} | '
          f'TTA + calibration {macro_f1(val_final, val_labels):.4f}')
    print(f'{name}: class weights {np.round(weights, 2)}')
    np.save(OUTPUT_DIR / f'class_weights_{name}.npy', weights)

    cm = confusion_matrix(val_labels, val_final.argmax(axis=1),
                          labels=np.arange(NUM_CLASSES), normalize='true')
    ConfusionMatrixDisplay(100 * cm, display_labels=CLASS_NAMES).plot(
        xticks_rotation='vertical', values_format='.1f')
    plt.title(f'{name}: validation confusion matrix after TTA + calibration (%)')
    plt.tight_layout()
    plt.show()
    return apply_class_weights(test_probs, weights).astype(np.float32)

## Validation numbers and per-class metrics

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

results = {}
for name, model in [('custom', customNet), ('ft', ftNet)]:
    val_plain, val_labels = predict_proba(model, val_dataloader, tta=False)
    val_tta, _ = predict_proba(model, val_dataloader, tta=True)
    weights, _ = fit_class_weights(val_tta, val_labels)
    val_cal = apply_class_weights(val_tta, weights)
    pred = val_cal.argmax(axis=1)
    p, r, f, s = precision_recall_fscore_support(
        val_labels, pred, labels=np.arange(NUM_CLASSES), zero_division=0)
    results[name] = {
        'source': custom_source if name == 'custom' else ft_source,
        'f1_plain': macro_f1(val_plain, val_labels),
        'f1_tta': macro_f1(val_tta, val_labels),
        'f1_cal': macro_f1(val_cal, val_labels),
        'acc_plain': accuracy_score(val_labels, val_plain.argmax(axis=1)),
        'acc_cal': accuracy_score(val_labels, pred),
        'class_weights': weights.tolist(),
        'precision': p.tolist(), 'recall': r.tolist(), 'f1': f.tolist(),
        'support': s.tolist(),
    }
    saved = CHECKPOINT_DIR / f'class_weights_{name}.npy'
    if saved.is_file():
        same = np.allclose(np.load(saved), weights)
        print(f'{name}: refitted calibration weights match the submitted ones: {same}')
    results[name]['_val_cal'] = val_cal
    print(f"{name}: val macro F1 plain {results[name]['f1_plain']:.4f} | "
          f"TTA {results[name]['f1_tta']:.4f} | TTA+cal {results[name]['f1_cal']:.4f} | "
          f"acc plain {results[name]['acc_plain']:.4f} -> {results[name]['acc_cal']:.4f}")
val_labels = np.asarray(val_dataset.labels)

(FIG_DIR / 'metrics.json').write_text(json.dumps(
    {k: {kk: vv for kk, vv in v.items() if not kk.startswith('_')} for k, v in results.items()},
    indent=2))

# LaTeX macros read by report/main.tex (copy this file to report/figures/).
macro_names = {'custom': 'Custom', 'ft': 'FT'}
lines = ['% Generated by report_figures.ipynb. Do not edit by hand.']
for name, m in results.items():
    tag = macro_names[name]
    lines += [rf'\renewcommand{{\{tag}ValPlain}}{{{m["f1_plain"]:.3f}}}',
              rf'\renewcommand{{\{tag}ValTTA}}{{{m["f1_tta"]:.3f}}}',
              rf'\renewcommand{{\{tag}ValCal}}{{{m["f1_cal"]:.3f}}}',
              rf'\renewcommand{{\{tag}AccPlain}}{{{m["acc_plain"]:.3f}}}',
              rf'\renewcommand{{\{tag}AccCal}}{{{m["acc_cal"]:.3f}}}']
(FIG_DIR / 'val_numbers.tex').write_text('\n'.join(lines) + '\n')

# Per-class table body: class & support & P/R/F1 custom & P/R/F1 FT.
rows = []
for c, cname in enumerate(CLASS_NAMES):
    cu, ft = results['custom'], results['ft']
    rows.append(f"{cname} & {cu['support'][c]} & {cu['precision'][c]:.2f} & {cu['recall'][c]:.2f} & "
                f"{cu['f1'][c]:.2f} & {ft['precision'][c]:.2f} & {ft['recall'][c]:.2f} & "
                f"{ft['f1'][c]:.2f} \\\\")
(FIG_DIR / 'per_class_rows.tex').write_text('\n'.join(rows) + '\n')
print('Wrote metrics.json, val_numbers.tex, per_class_rows.tex')

## Figure: confusion matrices

In [ ]:
# Drawn at its printed size (main.tex includes it at width=4.15in): fonts stay legible.
plt.rcParams.update({'font.size': 6.5})
fig, axes = plt.subplots(1, 2, figsize=(4.15, 2.1))
short = ['Water', 'V.Low', 'Low', 'Mod.', 'High', 'V.High', 'Non-b.']
for ax, (name, label) in zip(axes, [('custom', 'CUSTOM (residual CNN)'),
                                     ('ft', 'FINE-TUNING (ConvNeXt-T)')]):
    cm = 100 * confusion_matrix(val_labels, results[name]['_val_cal'].argmax(axis=1),
                                labels=np.arange(NUM_CLASSES), normalize='true')
    ax.imshow(cm, cmap='Blues', vmin=0, vmax=100)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, f'{cm[i, j]:.0f}', ha='center', va='center', fontsize=5.5,
                    color='white' if cm[i, j] > 55 else 'black')
    ax.set_xticks(range(NUM_CLASSES), short, rotation=45, ha='right')
    ax.set_yticks(range(NUM_CLASSES), short)
    ax.set_xlabel('Predicted class')
    ax.set_ylabel('True class')
    ax.set_title(f"{label}\nmacro F1 {results[name]['f1_cal']:.3f}", fontsize=6.5)
fig.tight_layout()
fig.savefig(FIG_DIR / 'fig_confusion.pdf', bbox_inches='tight')
plt.show()

## Figure: validation errors

In [ ]:
# Representative validation errors of each model: for the most frequent confusions,
# show the most confident wrong prediction (true label -> predicted label, probability).
def top_errors(val_cal, n=3):
    pred = val_cal.argmax(axis=1)
    cm = confusion_matrix(val_labels, pred, labels=np.arange(NUM_CLASSES))
    np.fill_diagonal(cm, 0)
    pairs = np.dstack(np.unravel_index(np.argsort(-cm, axis=None), cm.shape))[0]
    chosen = []
    for t, p in pairs:
        if cm[t, p] == 0 or len(chosen) == n:
            break
        idx = np.where((val_labels == t) & (pred == p))[0]
        chosen.append(idx[np.argmax(val_cal[idx, p])])
    return chosen, pred

fig, axes = plt.subplots(2, 3, figsize=(2.45, 2.15))  # printed at width=2.45in
for row, (name, label) in enumerate([('custom', 'CUSTOM'), ('ft', 'FT')]):
    chosen, pred = top_errors(results[name]['_val_cal'])
    for ax in axes[row]:
        ax.axis('off')
    for ax, i in zip(axes[row], chosen):
        ax.imshow(val_dataset.images[i])
        ax.set_title(f'{label} T: {CLASS_NAMES[val_labels[i]]}\nP: {CLASS_NAMES[pred[i]]} '
                     f'({results[name]["_val_cal"][i, pred[i]]:.2f})', fontsize=5)
fig.tight_layout(pad=0.15, h_pad=0.3, w_pad=0.2)
fig.savefig(FIG_DIR / 'fig_errors.pdf', bbox_inches='tight', dpi=200)
plt.show()

## Figure: learning curves

In [ ]:
if custom_history is None or ft_history is None:
    print('No learning curves: set RETRAIN_FOR_CURVES = True (GPU) to record them.')
else:
    fig, axes = plt.subplots(2, 2, figsize=(3.25, 1.85))  # printed at width=3.25in
    axes = axes.ravel()
    for k, (hist, label) in enumerate([(custom_history, 'CUSTOM'), (ft_history, 'FT')]):
        epochs = np.arange(1, len(hist['train_loss']) + 1)
        for phase, style in (('train', '-'), ('val', '--')):
            axes[2 * k].plot(epochs, hist[f'{phase}_loss'], style, label=phase)
            axes[2 * k + 1].plot(epochs, hist[f'{phase}_f1'], style, label=phase)
        best = int(np.argmax(hist['val_f1']))
        axes[2 * k + 1].axvline(best + 1, color='grey', lw=0.6)
        axes[2 * k].set_title(f'{label}: loss', fontsize=6.5)
        axes[2 * k + 1].set_title(f'{label}: macro F1 (best ep. {best + 1})', fontsize=6.5)
    for ax in axes:
        ax.set_xlabel('Epoch', fontsize=6, labelpad=1)
        ax.grid(alpha=0.3)
        ax.tick_params(labelsize=5.5, pad=1)
    axes[0].legend(fontsize=5.5)
    fig.tight_layout(pad=0.2, h_pad=0.3, w_pad=0.4)
    fig.savefig(FIG_DIR / 'fig_curves.pdf', bbox_inches='tight')
    plt.show()
    for name, hist in [('custom', custom_history), ('ft', ft_history)]:
        b = int(np.argmax(hist['val_f1']))
        print(f"{name} (retrain): best val F1 {hist['val_f1'][b]:.4f} at epoch {b + 1}, "
              f"train F1 there {hist['train_f1'][b]:.4f}")

## Package

In [ ]:
import shutil
archive = shutil.make_archive(str(OUTPUT_DIR / 'report_figures'), 'zip', FIG_DIR)
print('Copy these files into report/figures/ :')
for path in sorted(FIG_DIR.iterdir()):
    print('  ', path.name)
print('Zipped:', archive)